# 🧩 Parallelization: Sectioning and Voting

## Learning Objectives
In this notebook, you will learn:
1. **The two variants** - Sectioning (different subtasks at once) vs voting (the same task several times)
2. **Sectioning** - Generate independent parts of a story concurrently, each branch owning its own state key, then synthesize
3. **Measuring the speed-up** - Compare the parallel run's wall time with the sum of its branches
4. **Voting** - Run several independent reviewers in parallel and aggregate their verdicts with a threshold

## Where this fits
`01_parallelization_langgraph.ipynb` covered the graph mechanics — fan-out, fan-in, reducers — with toy nodes and a retrieval example. This notebook applies them to the two ways Anthropic's *Building Effective Agents* uses parallel LLM calls. `03_parallelization_lcel.ipynb` then shows the same mechanics in LCEL.

## Prerequisites
- Fan-out/fan-in and reducers — `01_parallelization_langgraph.ipynb`
- `with_structured_output` — `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (no other keys needed)

---
## 📖 Part 1: Two Variants of Parallelization

### The two variants

Parallelization is a powerful agentic workflow pattern that **divides complex tasks into independent processes that execute simultaneously**, dramatically reducing overall processing time and improving system throughput. This pattern leverages the principle that many computational tasks can be broken down into smaller, independent units that don't need to wait for each other to complete.

The parallelization pattern comes in two primary forms:

- **Sectioning**: Breaking down a task into independent subtasks that can run concurrently
- **Voting**: Running multiple processes simultaneously to cross-validate results and improve accuracy

### When to Use Parallelization

The parallelization pattern is most effective when:

- **Independent Subtasks**: Your workflow contains tasks that don't depend on each other's output
- **Latency Reduction**: You need to minimize total processing time for time-sensitive applications
- **Throughput Optimization**: You want to maximize the number of requests processed per unit time
- **Confidence Building**: Multiple parallel processes can validate results and increase reliability
- **Resource Utilization**: You have computational resources that can be leveraged simultaneously

| Property | Value |
|---|---|
| Origin | Anthropic, *Building Effective Agents* (Dec 2024) — [anthropic.com/research/building-effective-agents](https://www.anthropic.com/research/building-effective-agents), "Parallelization" section |

### Key Benefits

1. **Dramatically Reduced Latency**: Tasks that would take sequential time now complete in the time of the longest individual task
2. **Improved Resource Utilization**: Makes full use of available computational resources
3. **Enhanced Scalability**: System can handle more concurrent requests efficiently
4. **Fault Tolerance**: In voting scenarios, multiple processes provide redundancy
5. **Modular Design**: Independent processes are easier to test, debug, and maintain

---
## 📦 Step 1: Environment Setup

Load the API key, then create the chat model through the course factory.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


In [2]:
# ============================================================================
# LLM INITIALIZATION: One model for every branch
# ============================================================================
from aied_helpers import get_openai_llm

# temperature 0.8: story elements should be creative
story_generator = get_openai_llm(model="gpt-4o-mini", temperature=0.8)

print(f"✅ LLM initialized: {story_generator.model_name} (temperature={story_generator.temperature})")

✅ LLM initialized: gpt-4o-mini (temperature=0.8)


---
## ✂️ Part 2: Sectioning — Independent Parts of One Task

A story introduction needs characters, a setting and a premise. None of them depends on the others, so three branches can write them at the same time, and a fourth step weaves them together.

Notice the state: **each branch owns its own key** (`character_profiles`, `story_setting`, `plot_premise`). Because no two parallel nodes write the same key, no reducer is needed — compare the shared `context` list in `01_parallelization_langgraph.ipynb`, which did need one.

In [3]:
# ============================================================================
# STATE: One key per parallel branch, so no reducer is needed
# ============================================================================
from typing_extensions import TypedDict


class StoryGenerationState(TypedDict):
    story_topic: str        # Input topic for the story
    character_profiles: str # Generated character descriptions
    story_setting: str      # Generated setting description
    plot_premise: str       # Generated story premise
    final_introduction: str # Combined story introduction


print(f"✅ StoryGenerationState fields: {list(StoryGenerationState.__annotations__)}")

✅ StoryGenerationState fields: ['story_topic', 'character_profiles', 'story_setting', 'plot_premise', 'final_introduction']


### 2.1 🧱 The Three Section Writers and the Synthesizer

Each writer is a focused prompt for one section. The synthesizer is the fan-in point: it reads all three keys and asks for one cohesive opening.

In [4]:
# ============================================================================
# SECTION WRITERS: Three independent branches, one prompt each
# ============================================================================
def create_character_profiles(state: StoryGenerationState) -> dict:
    """Generate detailed character profiles for the story"""
    character_prompt = (
        f"Create 2-3 compelling characters for a story about {state['story_topic']}. "
        f"Include their names, key personality traits, and motivations. "
        f"Keep descriptions concise but vivid."
    )

    response = story_generator.invoke(character_prompt)
    return {"character_profiles": response.content}

def design_story_setting(state: StoryGenerationState) -> dict:
    """Generate an immersive story setting"""
    setting_prompt = (
        f"Describe a vivid, atmospheric setting for a story about {state['story_topic']}. "
        f"Include sensory details, mood, and environmental elements that enhance the narrative."
    )

    response = story_generator.invoke(setting_prompt)
    return {"story_setting": response.content}

def develop_plot_premise(state: StoryGenerationState) -> dict:
    """Generate a compelling story premise"""
    premise_prompt = (
        f"Write a compelling one-sentence plot premise for a story about {state['story_topic']}. "
        f"Include conflict, stakes, and intrigue to hook the reader."
    )

    response = story_generator.invoke(premise_prompt)
    return {"plot_premise": response.content}


# ============================================================================
# SYNTHESIZER: Fan-in — combine the three sections into one introduction
# ============================================================================
def synthesize_story_elements(state: StoryGenerationState) -> dict:
    """Combine all parallel-generated elements into a cohesive story introduction"""
    synthesis_prompt = (
        f"Create an engaging story introduction that seamlessly weaves together these elements:\n\n"
        f"CHARACTERS:\n{state['character_profiles']}\n\n"
        f"SETTING:\n{state['story_setting']}\n\n"
        f"PREMISE:\n{state['plot_premise']}\n\n"
        f"Write a compelling opening that introduces the characters, establishes the setting, "
        f"and hints at the central conflict. Keep it engaging and under 200 words."
    )

    response = story_generator.invoke(synthesis_prompt)
    return {"final_introduction": response.content}


print("✅ 3 section writers and 1 synthesizer defined")

✅ 3 section writers and 1 synthesizer defined


### 2.2 🏗️ Building the Sectioning Graph

Three edges out of `START` (fan-out) and three edges into `synthesize_elements` (fan-in).

In [5]:
# ============================================================================
# BUILD THE GRAPH: START → 3 writers in parallel → synthesizer → END
# ============================================================================
from langgraph.graph import END, START, StateGraph

parallel_story_graph = StateGraph(StoryGenerationState)

# Add all processing nodes
parallel_story_graph.add_node("create_characters", create_character_profiles)
parallel_story_graph.add_node("design_setting", design_story_setting)
parallel_story_graph.add_node("develop_premise", develop_plot_premise)
parallel_story_graph.add_node("synthesize_elements", synthesize_story_elements)

# Configure parallel execution: all three generators start simultaneously
parallel_story_graph.add_edge(START, "create_characters")
parallel_story_graph.add_edge(START, "design_setting")
parallel_story_graph.add_edge(START, "develop_premise")

# All parallel processes feed into the synthesis step
parallel_story_graph.add_edge("create_characters", "synthesize_elements")
parallel_story_graph.add_edge("design_setting", "synthesize_elements")
parallel_story_graph.add_edge("develop_premise", "synthesize_elements")

# Final step completes the workflow
parallel_story_graph.add_edge("synthesize_elements", END)

# Compile the parallel workflow
story_workflow = parallel_story_graph.compile()

print("✅ Sectioning workflow compiled")

✅ Sectioning workflow compiled


In [6]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = story_workflow.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

🧩 Nodes: ['create_characters', 'design_setting', 'develop_premise', 'synthesize_elements']
🔗 Edges:
            __start__ ──▶ create_characters  (fixed)
            __start__ ──▶ design_setting     (fixed)
            __start__ ──▶ develop_premise    (fixed)
    create_characters ──▶ synthesize_elements (fixed)
       design_setting ──▶ synthesize_elements (fixed)
      develop_premise ──▶ synthesize_elements (fixed)
   synthesize_elements ──▶ __end__            (fixed)


---
## ⏱️ Part 3: Running It and Measuring the Speed-Up

`stream_mode="updates"` yields one event per finished node, so we can time each section writer as it completes. All three start at time zero, so each finish time is also that branch's duration: the parallel step takes as long as the *slowest* branch, while a sequential chain would take the *sum*.

In [7]:
# ============================================================================
# RUN: Stream node updates and time when each branch finishes
# ============================================================================
import time

topic = "time-traveling detective"
print(f"📖 Generating story for: {topic}\n")

start = time.perf_counter()
finished_at = {}
result = {}
for update in story_workflow.stream({"story_topic": topic}, stream_mode="updates"):
    for node, values in update.items():
        finished_at[node] = time.perf_counter() - start
        result.update(values)
        print(f"  ✅ {node:<20} finished at {finished_at[node]:.1f}s")

branches = ["create_characters", "design_setting", "develop_premise"]
parallel_step = max(finished_at[b] for b in branches)
sequential_estimate = sum(finished_at[b] for b in branches)
print(f"\n⏱️  Parallel sections step:        {parallel_step:.1f}s (the slowest branch)")
print(f"⏱️  Same three calls in sequence: ~{sequential_estimate:.1f}s (the sum)")
print(f"⏱️  Total with synthesis:         {finished_at['synthesize_elements']:.1f}s")
print(f"\n📄 Final Story Introduction:\n{result['final_introduction']}")

📖 Generating story for: time-traveling detective

  ✅ develop_premise      finished at 1.6s
  ✅ create_characters    finished at 4.7s
  ✅ design_setting       finished at 7.0s
  ✅ synthesize_elements  finished at 10.6s

⏱️  Parallel sections step:        7.0s (the slowest branch)
⏱️  Same three calls in sequence: ~13.4s (the sum)
⏱️  Total with synthesis:         10.6s

📄 Final Story Introduction:
In the heart of a city where the past and present entwined like lovers in a dance, Detective Marlowe Voss stood at the threshold of a forgotten alley, the mist wrapping around him like a shroud. His trench coat flared in the chill evening breeze, each raindrop on the cobblestones echoing the unsolved murder of his closest friend—a ghost that haunted him relentlessly. 

Beside him, Lila Quinn bounced on her heels, her curious eyes glinting with the thrill of the unknown. “What if we find out my grandfather really was innocent?” she mused, her tech-savvy mind already racing ahead. Only a few pa

Read the timestamps: the section writers finish at different moments because their outputs differ in length, but they all started together, so the synthesizer began as soon as the slowest one finished — not after the sum of all three. The synthesis call itself cannot be parallelised: it needs every section.

---
## 🗳️ Part 4: Voting — The Same Task, Several Times

Voting runs the **same kind of check** several times in parallel and aggregates the results. The Anthropic example is a code review for vulnerabilities, where several different prompts each review the code and flag a problem; the verdict needs a threshold of flags to fire.

Here three reviewers look at the same function from different angles (injection, secrets handling, input validation). Each returns a structured `Vote`. They all write to one `votes` list — so this time a **reducer is required**, exactly as in Part 2 of `01_parallelization_langgraph.ipynb`.

### Key Concepts:
- **Independent voters**: each reviewer sees only the code, never another reviewer's vote
- **Threshold**: the aggregator decides how many flags are needed (here 2 of 3) — a knob that trades false alarms against misses

In [8]:
# ============================================================================
# VOTE SCHEMA AND STATE: every reviewer appends one Vote to a shared list
# ============================================================================
import operator
from typing import Annotated

from pydantic import BaseModel, Field


class Vote(BaseModel):
    """One reviewer's verdict on the code."""

    flagged: bool = Field(description="True if this reviewer found a security problem")
    reason: str = Field(description="One sentence explaining the verdict")


class ReviewState(TypedDict):
    code: str
    votes: Annotated[list, operator.add]  # parallel writes → reducer required
    verdict: str


FLAG_THRESHOLD = 2  # flags needed (out of 3) to block the code

# temperature 0: a reviewer should give the same verdict on the same code
reviewer_llm = get_openai_llm(model="gpt-4o-mini").with_structured_output(Vote)

print("✅ Vote schema and ReviewState defined")

✅ Vote schema and ReviewState defined


In [9]:
# ============================================================================
# REVIEWERS AND AGGREGATOR: three perspectives, one threshold
# ============================================================================
PERSPECTIVES = {
    "injection_review": "SQL, shell or code injection through unsanitised input",
    "secrets_review": "hard-coded credentials, tokens or keys, and leaking secrets in logs",
    "validation_review": "missing input validation or unsafe handling of user-controlled values",
}


def make_reviewer(name: str, focus: str):
    """Build a reviewer node that only looks for one class of problem."""

    def review(state: ReviewState) -> dict:
        vote = reviewer_llm.invoke(
            f"You are a security reviewer. Only look for: {focus}. "
            f"Flag the code only if you find that kind of problem.\n\n```python\n{state['code']}\n```"
        )
        print(f"  {'🚩' if vote.flagged else '✅'} {name:<18} {vote.reason}")
        return {"votes": [{"reviewer": name, "flagged": vote.flagged}]}

    return review


def aggregate_votes(state: ReviewState) -> dict:
    """Fan-in: count the flags and apply the threshold."""
    flags = sum(v["flagged"] for v in state["votes"])
    verdict = "BLOCK" if flags >= FLAG_THRESHOLD else "APPROVE"
    return {"verdict": f"{verdict} ({flags}/{len(state['votes'])} reviewers flagged)"}


review_builder = StateGraph(ReviewState)
for name, focus in PERSPECTIVES.items():
    review_builder.add_node(name, make_reviewer(name, focus))
    review_builder.add_edge(START, name)                 # fan-out
    review_builder.add_edge(name, "aggregate_votes")     # fan-in
review_builder.add_node("aggregate_votes", aggregate_votes)
review_builder.add_edge("aggregate_votes", END)

review_workflow = review_builder.compile()

print("✅ Voting workflow compiled:", [n for n in review_workflow.get_graph().nodes if not n.startswith("__")])

✅ Voting workflow compiled: ['injection_review', 'secrets_review', 'validation_review', 'aggregate_votes']


### 4.1 🧪 Two Snippets, Two Verdicts

The first snippet builds SQL from user input and hard-codes a password — several reviewers should flag it. The second uses a parameterised query and reads its secret from the environment — it should pass.

In [10]:
# ============================================================================
# RUN: One unsafe and one safe snippet
# ============================================================================
UNSAFE_CODE = """
DB_PASSWORD = "hunter2"

def get_user(conn, username):
    query = f"SELECT * FROM users WHERE name = '{username}'"
    print(f"connecting with password {DB_PASSWORD}")
    return conn.execute(query).fetchall()
"""

SAFE_CODE = """
import os

def get_user(conn, username: str):
    if not username.isalnum() or len(username) > 32:
        raise ValueError("invalid username")
    password = os.environ["DB_PASSWORD"]  # never logged
    return conn.execute("SELECT * FROM users WHERE name = ?", (username,)).fetchall()
"""

for label, snippet in [("unsafe", UNSAFE_CODE), ("safe", SAFE_CODE)]:
    print(f"\n🔍 Reviewing the {label} snippet:")
    outcome = review_workflow.invoke({"code": snippet})
    print(f"⚖️  Verdict: {outcome['verdict']}")


🔍 Reviewing the unsafe snippet:
  🚩 secrets_review     The code contains a hard-coded database password.
  🚩 validation_review  The code uses string interpolation to construct a SQL query, making it vulnerable to SQL injection attacks.
  🚩 injection_review   The code uses unsanitized input in the SQL query, making it vulnerable to SQL injection.
⚖️  Verdict: BLOCK (3/3 reviewers flagged)

🔍 Reviewing the safe snippet:
  ✅ injection_review   The code uses parameterized queries, which prevent SQL injection.
  ✅ secrets_review     No hard-coded credentials, tokens, or keys found, and secrets are not logged.
  ✅ validation_review  The code properly validates the username input and uses parameterized queries to prevent SQL injection.
⚖️  Verdict: APPROVE (0/3 reviewers flagged)


Each reviewer judged only its own concern, and the aggregator — plain Python — made the call. Raising `FLAG_THRESHOLD` to 3 makes the gate stricter about blocking (fewer false alarms, more misses); lowering it to 1 blocks on any single flag.

---
## 📝 Summary

In this notebook, we learned:

### 1. The two variants
- **Sectioning** splits one task into different, independent subtasks; **voting** runs the same task several times and aggregates

### 2. Sectioning
- One focused prompt per section, one synthesizer as the fan-in
- When each branch owns its own state key, no reducer is needed

### 3. Measuring the speed-up
- `stream_mode="updates"` shows when each node finishes; the parallel step lasts as long as its slowest branch, not the sum

### 4. Voting
- Independent voters append to one list, so the key needs a reducer
- The aggregator's threshold is a design decision: it trades false alarms against misses

### Next Steps
- **`03_parallelization_lcel.ipynb`**: the same fan-out/fan-in mechanics in LCEL with `RunnableParallel`, plus concurrency limits